In [ ]:
class VendingMachine:
    def __init__(self):
        # 상품 정보 (상품명: [가격, 재고])
        self.products = {
            "1": ["콜라", 1500, 10],
            "2": ["사이다", 1300, 8],
            "3": ["커피", 2000, 5],
            "4": ["물", 1000, 15],
            "5": ["오렌지주스", 1800, 7],
            "6": ["초콜릿", 2500, 3]
        }
        
        # 투입된 금액
        self.inserted_money = 0
        
        # 거스름돈용 동전/지폐 보유량
        self.change_stock = {
            10: 50,     # 10원 50개
            50: 30,     # 50원 30개  
            100: 40,    # 100원 40개
            500: 20,    # 500원 20개
            1000: 10    # 1000원 10개
        }
    
    def display_products(self):
        print("\n" + "="*50)
        print("           🥤 자판기 상품 목록 🥤")
        print("="*50)
        for code, (name, price, stock) in self.products.items():
            status = "✅" if stock > 0 else "❌ 품절"
            print(f"{code}. {name:<10} {price:>5}원  재고: {stock:>2}개 {status}")
        print("="*50)
        print(f"💰 투입된 금액: {self.inserted_money:,}원")
        print("="*50)
    
    def insert_money(self):
        print("\n💸 돈을 투입해주세요 (10, 50, 100, 500, 1000원만 가능)")
        print("0을 입력하면 메뉴로 돌아갑니다.")
        
        while True:
            try:
                money = int(input("투입할 금액: "))
                if money == 0:
                    break
                elif money in [10, 50, 100, 500, 1000]:
                    self.inserted_money += money
                    print(f"✅ {money}원이 투입되었습니다. 총 투입금액: {self.inserted_money:,}원")
                else:
                    print("❌ 10, 50, 100, 500, 1000원만 투입 가능합니다.")
            except ValueError:
                print("❌ 숫자만 입력해주세요.")
    
    def calculate_change(self, change_amount):
        """거스름돈 계산 및 반환"""
        change_coins = {}
        remaining = change_amount
        
        # 큰 단위부터 거스름돈 계산
        for coin in sorted(self.change_stock.keys(), reverse=True):
            if remaining >= coin and self.change_stock[coin] > 0:
                needed = min(remaining // coin, self.change_stock[coin])
                if needed > 0:
                    change_coins[coin] = needed
                    remaining -= coin * needed
                    self.change_stock[coin] -= needed
        
        return change_coins, remaining
    
    def buy_product(self):
        if self.inserted_money == 0:
            print("❌ 먼저 돈을 투입해주세요!")
            return
        
        print("\n🛒 구매할 상품 번호를 입력하세요:")
        product_code = input("상품 번호: ")
        
        if product_code not in self.products:
            print("❌ 잘못된 상품 번호입니다.")
            return
        
        name, price, stock = self.products[product_code]
        
        if stock <= 0:
            print(f"❌ {name}은(는) 품절입니다.")
            return
        
        if self.inserted_money < price:
            shortage = price - self.inserted_money
            print(f"❌ 돈이 부족합니다. {shortage:,}원이 더 필요합니다.")
            return
        
        # 구매 처리
        change_amount = self.inserted_money - price
        self.products[product_code][2] -= 1  # 재고 감소
        
        # 거스름돈 계산
        change_coins, remaining_change = self.calculate_change(change_amount)
        
        if remaining_change > 0:
            print(f"❌ 거스름돈이 부족합니다. ({remaining_change}원 부족)")
            # 구매 취소 - 재고 복원
            self.products[product_code][2] += 1
            return
        
        # 구매 완료
        print(f"\n🎉 구매 완료!")
        print(f"📦 상품: {name}")
        print(f"💰 결제금액: {price:,}원")
        
        if change_amount > 0:
            print(f"💵 거스름돈: {change_amount:,}원")
            print("거스름돈 내역:")
            for coin, count in sorted(change_coins.items(), reverse=True):
                print(f"  {coin}원 x {count}개")
        
        self.inserted_money = 0  # 투입금액 초기화
        print("상품이 나왔습니다! 🥤")
    
    def return_money(self):
        """투입된 돈 반환"""
        if self.inserted_money == 0:
            print("❌ 투입된 돈이 없습니다.")
            return
        
        change_coins, remaining = self.calculate_change(self.inserted_money)
        
        if remaining > 0:
            print(f"❌ 거스름돈이 부족합니다. {remaining}원을 반환할 수 없습니다.")
            # 거스름돈 복원
            for coin, count in change_coins.items():
                self.change_stock[coin] += count
            return
        
        print(f"\n💵 {self.inserted_money:,}원이 반환됩니다.")
        print("반환 내역:")
        for coin, count in sorted(change_coins.items(), reverse=True):
            print(f"  {coin}원 x {count}개")
        
        self.inserted_money = 0
    
    def admin_menu(self):
        """관리자 메뉴"""
        password = input("관리자 비밀번호를 입력하세요: ")
        if password != "admin123":
            print("❌ 잘못된 비밀번호입니다.")
            return
        
        while True:
            print("\n" + "="*30)
            print("      🔧 관리자 메뉴")
            print("="*30)
            print("1. 재고 보충")
            print("2. 상품 가격 변경") 
            print("3. 거스름돈 보충")
            print("4. 매출 확인")
            print("0. 돌아가기")
            
            choice = input("선택: ")
            
            if choice == "1":
                self.restock_products()
            elif choice == "2":
                self.change_price()
            elif choice == "3":
                self.restock_change()
            elif choice == "4":
                self.show_sales()
            elif choice == "0":
                break
            else:
                print("❌ 잘못된 선택입니다.")
    
    def restock_products(self):
        """상품 재고 보충"""
        self.display_products()
        product_code = input("\n재고를 보충할 상품 번호: ")
        
        if product_code not in self.products:
            print("❌ 잘못된 상품 번호입니다.")
            return
        
        try:
            amount = int(input("보충할 수량: "))
            if amount > 0:
                self.products[product_code][2] += amount
                name = self.products[product_code][0]
                print(f"✅ {name} 재고가 {amount}개 보충되었습니다.")
            else:
                print("❌ 양수만 입력해주세요.")
        except ValueError:
            print("❌ 숫자만 입력해주세요.")
    
    def change_price(self):
        """상품 가격 변경"""
        self.display_products()
        product_code = input("\n가격을 변경할 상품 번호: ")
        
        if product_code not in self.products:
            print("❌ 잘못된 상품 번호입니다.")
            return
        
        try:
            new_price = int(input("새로운 가격: "))
            if new_price > 0:
                old_price = self.products[product_code][1]
                self.products[product_code][1] = new_price
                name = self.products[product_code][0]
                print(f"✅ {name} 가격이 {old_price}원 → {new_price}원으로 변경되었습니다.")
            else:
                print("❌ 양수만 입력해주세요.")
        except ValueError:
            print("❌ 숫자만 입력해주세요.")
    
    def restock_change(self):
        """거스름돈 보충"""
        print("\n현재 거스름돈 보유량:")
        for coin, count in sorted(self.change_stock.items(), reverse=True):
            print(f"{coin}원: {count}개")
        
        try:
            coin = int(input("\n보충할 동전/지폐 종류 (10, 50, 100, 500, 1000): "))
            if coin in self.change_stock:
                amount = int(input(f"{coin}원 보충할 개수: "))
                if amount > 0:
                    self.change_stock[coin] += amount
                    print(f"✅ {coin}원이 {amount}개 보충되었습니다.")
                else:
                    print("❌ 양수만 입력해주세요.")
            else:
                print("❌ 올바른 동전/지폐 종류를 입력해주세요.")
        except ValueError:
            print("❌ 숫자만 입력해주세요.")
    
    def show_sales(self):
        """매출 정보 표시"""
        print("\n📊 현재 상품별 재고 현황:")
        total_value = 0
        for code, (name, price, stock) in self.products.items():
            value = price * stock
            total_value += value
            print(f"{name}: {stock}개 (가치: {value:,}원)")
        
        print(f"\n💰 총 재고 가치: {total_value:,}원")
        
        change_value = sum(coin * count for coin, count in self.change_stock.items())
        print(f"💵 거스름돈 보유액: {change_value:,}원")
    
    def run(self):
        """자판기 실행"""
        print("🥤 자판기가 시작되었습니다! 🥤")
        
        while True:
            print("\n" + "="*40)
            print("           🥤 자판기 메뉴")
            print("="*40)
            print("1. 상품 보기")
            print("2. 돈 투입")
            print("3. 상품 구매")
            print("4. 돈 반환")
            print("9. 관리자 메뉴")
            print("0. 종료")
            print("="*40)
            
            choice = input("메뉴를 선택하세요: ")
            
            if choice == "1":
                self.display_products()
            elif choice == "2":
                self.insert_money()
            elif choice == "3":
                self.buy_product()
            elif choice == "4":
                self.return_money()
            elif choice == "9":
                self.admin_menu()
            elif choice == "0":
                print("자판기를 종료합니다. 안녕히 가세요! 👋")
                break
            else:
                print("❌ 잘못된 선택입니다. 다시 선택해주세요.")


# 자판기 실행
if __name__ == "__main__":
    vending_machine = VendingMachine()
    vending_machine.run()